In [15]:
from pyspark.sql.functions import (
    sum,
    avg,
    countDistinct,
    year,
    month,
    col
)

StatementMeta(SparkProyecto, 11, 16, Finished, Available, Finished, False)

### Leer Silver

In [1]:
dim_customer = spark.table("silver.dim_customer")
dim_product  = spark.table("silver.dim_product")
fact_sales   = spark.table("silver.fact_sales")

StatementMeta(, , -1, SessionStarting, , SessionStarting, True)

### GOLD 1 - Ventas por mes

In [17]:
sales_by_month = (
    fact_sales
    .groupBy(
        year("OrderDate").alias("Year"),
        month("OrderDate").alias("Month")
    )
    .agg(
        sum("LineTotal").alias("TotalSales")
    )
)


StatementMeta(SparkProyecto, 11, 18, Finished, Available, Finished, False)

In [18]:
sales_by_month.write \
    .mode("overwrite") \
    .saveAsTable("gold.sales_by_month")

StatementMeta(SparkProyecto, 11, 19, Finished, Available, Finished, False)

In [19]:
sales_by_month.write \
    .mode("overwrite") \
    .parquet(
        "abfss://adlsdatamau@sadp203mau.dfs.core.windows.net/03-Gold/sales_by_month"
    )

StatementMeta(SparkProyecto, 11, 20, Finished, Available, Finished, False)

### GOLD 2 - Ventas por categoría

In [20]:
sales_by_category = (
    fact_sales.alias("f")
    .join(
        dim_product.alias("p"),
        "ProductID"
    )
    .groupBy("CategoryName")
    .agg(
        sum("LineTotal").alias("TotalSales")
    )
)


StatementMeta(SparkProyecto, 11, 21, Finished, Available, Finished, False)

In [21]:
sales_by_category.write \
    .mode("overwrite") \
    .saveAsTable("gold.sales_by_category")

StatementMeta(SparkProyecto, 11, 22, Finished, Available, Finished, False)

In [22]:
sales_by_category.write \
    .mode("overwrite") \
    .parquet(
        "abfss://adlsdatamau@sadp203mau.dfs.core.windows.net/03-Gold/sales_by_category"
    )

StatementMeta(SparkProyecto, 11, 23, Finished, Available, Finished, False)

### GOLD 3 - Top productos

In [23]:
top_products = (
    fact_sales.alias("f")
    .join(
        dim_product.alias("p"),
        "ProductID"
    )
    .groupBy(
        "ProductID",
        "ProductName"
    )
    .agg(
        sum("LineTotal").alias("SalesAmount")
    )
    .orderBy(
        col("SalesAmount").desc()
    )
)

StatementMeta(SparkProyecto, 11, 24, Finished, Available, Finished, False)

In [24]:
top_products.write \
    .mode("overwrite") \
    .saveAsTable("gold.top_products")

StatementMeta(SparkProyecto, 11, 25, Finished, Available, Finished, False)

In [25]:
top_products.write \
    .mode("overwrite") \
    .parquet(
        "abfss://adlsdatamau@sadp203mau.dfs.core.windows.net/03-Gold/top_products"
    )

StatementMeta(SparkProyecto, 11, 26, Finished, Available, Finished, False)

### GOLD 4 - KPIs ejecutivos

In [26]:
sales_kpis = (
    fact_sales
    .agg(
        countDistinct("CustomerID").alias("Customers"),
        countDistinct("SalesOrderID").alias("Orders"),
        sum("LineTotal").alias("Revenue"),
        avg("LineTotal").alias("AverageTicket")
    )
)

StatementMeta(SparkProyecto, 11, 27, Finished, Available, Finished, False)

In [27]:
sales_kpis.write \
    .mode("overwrite") \
    .saveAsTable("gold.sales_kpis")

StatementMeta(SparkProyecto, 11, 28, Finished, Available, Finished, False)

In [28]:
sales_kpis.write \
    .mode("overwrite") \
    .parquet(
        "abfss://adlsdatamau@sadp203mau.dfs.core.windows.net/03-Gold/sales_kpis"
    )

StatementMeta(SparkProyecto, 11, 29, Finished, Available, Finished, False)